<a href="https://colab.research.google.com/github/fbsilvaRP/ComputerVision-Fundamentals/blob/main/RedesNeurais_Estruturado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Célula de SETUP 1 -  Organização do ambiente e extração de dados**


---
A célula abaixo realiza as seguintes etapas:

1.   Testes da GPU (Verificação);
2.   Importação da base de dados EBHI do google drive.

**Observação:** sempre rodar a célula no início de uma nova sessão do Colab





In [2]:
import torch
import os
import shutil
from google.colab import drive

# --- Seção de apresentação inicial ---
print("=-" * 30)
print("Primeira célula de setup - Versão 1")
print("Última atualização em 12 de Agosto de 2026")
print("=-" * 30)

# ---1. Verificação da GPU ---
print("Etapa 1: Verificação da GPU")
print("-=" * 30)

print("Versão do PyTorch: ", torch.__version__) #Versão do PyTorch em uso;

resposta = torch.cuda.is_available()            #Verificação se a GPU está disponível para uso (1 ou 0);

if(resposta):
  nomeGPU = torch.cuda.get_device_name(0)       #Verifica o nome da GPU em uso;
  print(f"Nome da GPU: {nomeGPU}")              #Apresenta o nome obtido;
  device = torch.device('cuda')                 #Atribui a identificação para a GPU;
else:
  print("GPU não disponível. Usando CPU.")
  device = torch.device('cpu')                  #Atribui a identificação para a CPU;

# --- 2.Importando a base de imagens EBHI ---

!apt-get install unrar -y                       #Instala o unrar a partir da linha de comando (comando do linux);

drive.mount('/content/drive')                   #Estabelece a conexão com o Google Drive;

origem_rar  =   '/content/drive/MyDrive/EBH-HE-IDS.rar' #Definição da origem da base de imagens (arquivo rar);
destino_rar =   '/content/EBHI.rar'                     #Definição de onde o arquivo (rar) será colocado - Voltar um diretório;
pasta_destino = '/content/EBHI'                         #Pasta onde o arquivo será extraído;

if os.path.exists(pasta_destino):                       #Se o diretório já existir, ele será apagado;
  shutil.rmtree(pasta_destino)                          #Apaga um diretório preeexistente, se existir;
  print("Arquivo preexistente apagado.")


shutil.copy(origem_rar, destino_rar)                  #Cópia do Drive para a máquina local do Colab;

!unrar x {destino_rar} {pasta_destino}/           #Realiza a descompactação do arquivo (comando linux);

print("Configuração concluída!")                      #Mensagem de conclusão;

A saída de streaming foi truncada nas últimas 5000 linhas.
Extracting  /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/100/2016780-1-100-005.jpg      43%  OK 
Extracting  /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/100/2016780-1-100-006.jpg      43%  OK 
Extracting  /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/100/2016780-1-100-007.jpg      43%  OK 
Creating    /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/200  OK
Extracting  /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/200/2016780-1-200-001.jpg      43%  OK 
Extracting  /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/200/2016780-1-200-002.jpg      43%  OK 
Extracting  /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/200/2016780-1-200-003.jpg      43%  OK 
Creating    /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/40  OK
Extracting  /content/EBHI/ColHis-IDS/High-grade IN/20210115-2016780-1/40

**Célula de SETUP 2 - Criação do dataframe**


---
O dataframe funcionará como uma planilha.

1.   Criação do dataframe;
2.   Apresentação das quantidades de imagens + proporção atual de imagens


**Observação:** Para a classificação a ser realizada, será utilizada apenas a **magnificação 200x** (sugestão do artigo).




In [34]:
import os
import pandas as pd

caminho_base = '/content/EBHI/ColHis-IDS' #Define a pasta onde a busca por imagens será feita

registros = [] #Lista vazia (futuro dataframe)

for pasta_atual, subpastas, arquivos in os.walk(caminho_base):    #os.walk() função que percorre todos os diretórios a partir do caminho base;
  for arquivo in arquivos:
    #Dentro de cada pasta visitada, o loop passa por cada arquivo individualmente
    if arquivo.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):     #Filtra para processar só arquivos de imagem;
      #endswith verifica se o nome termina com qualquer uma das extensões da tupla
      caminho_completo = os.path.join(pasta_atual, arquivo)             #Junta o caminho da pasta atual com o nome do arquivo, formando o caminho completo até o arquivo específico;

      partes = caminho_completo.replace(caminho_base, '').strip(os.sep).split(os.sep)

      if len(partes) == 4:
        classe, paciente, magnificacao, nome_arquivo = partes
        registros.append({
            'caminho': caminho_completo,
            'classe_5': classe,
            'paciente': paciente,
            'magnificacao': magnificacao,
            'arquivo': nome_arquivo
        })

df = pd.DataFrame(registros)

#Inserindo os dados obtidos em variáveis (p/apresentar)
totalImagens = len(df)
totalClasses = df['classe_5'].value_counts()
Mag = df['magnificacao'].value_counts()
TotalPacientes = df['paciente'].nunique()

# --- Apresentando o conteúdo obtido ---

print("=" * 50)
print("RESUMO DO DATAFRAME MASTER — BASE EBHI")
print("=" * 50)

print(f"\nTotal de imagens: {totalImagens}")

print(f"\n{'Classes encontradas':-^40}")

for classe, quantidade in totalClasses.items():
    percentual = (quantidade / totalImagens) * 100
    print(f"  {classe:<20} {quantidade:>5}  ({percentual:5.1f}%)")

print(f"\n{'Magnificações encontradas':-^40}")

for magnificacao, quantidade in Mag.items():
    percentual = (quantidade / totalImagens) * 100
    print(f"  {magnificacao + 'x':<20} {quantidade:>5}  ({percentual:5.1f}%)")

print(f"\nTotal de pacientes/lâminas únicas: {TotalPacientes}")
print("=" * 50)

# tabela cruzada: linhas = classe, colunas = magnificação, valores = contagem
tabela_cruzada = pd.crosstab(
    df['classe_5'],
    df['magnificacao'],
    margins= True,           # adiciona linha e coluna de "Total"
    margins_name= 'Total'
)

# reordena as colunas para ficar 40, 100, 200, 400, Total (mesma ordem do artigo)
ordem_colunas = ['40', '100', '200', '400', 'Total']
tabela_cruzada = tabela_cruzada[ordem_colunas]

# reordena as linhas para bater com a ordem do artigo (Normal, Polyp, Low-grade IN, High-grade IN, Adenocarcinoma, Total)
ordem_linhas = ['Normal', 'Polyp', 'Low-grade IN', 'High-grade IN', 'Adenocarcinoma', 'Total']
tabela_cruzada = tabela_cruzada.reindex(ordem_linhas)

print(f"\n{'Tabela com os dados cruzados(Artigo)':-^50}")
print("=" * 50)

print(tabela_cruzada)
print("=" * 50)

RESUMO DO DATAFRAME MASTER — BASE EBHI

Total de imagens: 5532

----------Classes encontradas-----------
  Adenocarcinoma        2278  ( 41.2%)
  Low-grade IN          1808  ( 32.7%)
  Polyp                  842  ( 15.2%)
  High-grade IN          418  (  7.6%)
  Normal                 186  (  3.4%)

-------Magnificações encontradas--------
  400x                  2016  ( 36.4%)
  200x                  1838  ( 33.2%)
  100x                  1086  ( 19.6%)
  40x                    592  ( 10.7%)

Total de pacientes/lâminas únicas: 507

-------Tabela com os dados cruzados(Artigo)-------
magnificacao     40   100   200   400  Total
classe_5                                    
Normal           17    29    61    79    186
Polyp           119   165   254   304    842
Low-grade IN    204   341   603   660   1808
High-grade IN    47    80   130   161    418
Adenocarcinoma  205   471   790   812   2278
Total           592  1086  1838  2016   5532
